In [ ]:
# CLEANING & TRANSATION

In [ ]:
##User Data Exploration Summary

# We checked the data structure and columns first.
# We fixed timestamp formats to make them readable and usable for analysis.
# We kept user_id as it is because it is a unique identifier.
# Country values were standardized (e.g. unified naming like "KSA").
# Email can be NULL and this is acceptable in the system.
# Last login can also be NULL if the user never logged in.
# We created simple flags:
# has_email (1 if email exists, 0 if NULL)
# has_logged_in (1 if last_login exists, 0 if NULL)
# We found some invalid cases:
# If last_login < created_at, we treat it as invalid data and marked it using an last_login_invalid_flag .
# We mark it using a flag instead of deleting the row.
# We also added a review flag for edge cases like:
# email is NULL but last_login exists.
# We did not delete any records, only cleaned and marked issues.
# The goal was to keep raw data safe and create a clean version for analysis.

In [ ]:
# user_silver_transformation= """
# SELECT
#     user_id,
#     email,
#     COALESCE(
#         MAP('KSA', 'Saudi Arabia',
#             'ECY', 'Egypt',
#             'UAE', 'United Arab Emirates',
#             'QAT', 'Qatar',
#             'KWT', 'Kuwait',
#             'jOR', 'Jordan',
#             'MAR', 'Morocco'
#         )[country],
#         'Unknown'
#     ) AS country_name,
#     account_status,
#     CAST(created_at / 1000000000 AS TIMESTAMP) AS created_at,
#     CAST(last_login / 1000000000 AS TIMESTAMP) AS last_login,
#     CASE WHEN email IS NOT NULL THEN 1 ELSE 0 END AS has_email,
#     CASE WHEN last_login IS NOT NULL THEN 1 ELSE 0 END AS has_logged_in,
#     CASE WHEN last_login < created_at THEN NULL ELSE last_login END AS last_login_clean,
#     CASE WHEN last_login < created_at THEN 1 ELSE 0 END AS last_login_invalid_flag,
#     CASE WHEN email IS NULL AND last_login IS NOT NULL THEN "needs_review" ELSE NULL
#     END AS review_flag,
#     CAST(CAST(created_at / 1000000000 AS TIMESTAMP)AS DATE) AS created_date,
#     CAST(CAST(last_login / 1000000000 AS TIMESTAMP)AS DATE) AS last_login_date
# FROM users_table
# """

In [ ]:
#usres_df = spark.sql(user_silver_transformation)

In [ ]:
# subscription_silver_transformation= """
# SELECT
#     subscription_id,
#     user_id,
#     plan_type,
#     CAST(start_date / 1000000000 AS TIMESTAMP) AS start_date,
#     CAST(end_date / 1000000000 AS TIMESTAMP) AS end_date,
#     CASE WHEN end_date < start_date THEN 1 ELSE end_date END AS end_date_clean,
#     CASE WHEN end_date < start_date THEN 1 ELSE 0 END AS end_date_invalid_flag,
#     CASE WHEN end_date IS NOT NULL AND end_date < start_date THEN "needs_review" ELSE NULL END AS review_flag,
#     is_active,
#     CAST(CAST(start_date / 1000000000 AS TIMESTAMP)AS DATE) AS created_date,
#     CAST(CAST(end_date / 1000000000 AS TIMESTAMP)AS DATE) AS last_login_date
# FROM subscriptions_table
# """

In [ ]:
#subscriptions_df = spark.sql(subscription_silver_transformation)

In [ ]:
# payments_silver_transformation= """
# SELECT
#     payment_id,
#     subscription_id,
#     CASE
#         WHEN payment_status = "success" THEN ABS(amount)
#         WHEN payment_status = "failed" THEN 0
#     END AS amount,
#     currency,
#     CAST(payment_date / 1000000000 AS TIMESTAMP) AS payment_date,
#     CAST(CAST(payment_date/ 1000000000 AS TIMESTAMP)AS DATE) AS payment_short_date,
#     payment_status
# FROM payments_table
# """

In [ ]:
#payments_df = spark.sql(payments_silver_transformation)

In [ ]:
# BUSINESS LOGIC
# OUTPUT